In [1]:
import torch

from sentence_transformers import SentenceTransformer, SentenceTransformerModelCardData, SentenceTransformerTrainingArguments, SentenceTransformerTrainer
from sentence_transformers.evaluation import InformationRetrievalEvaluator, SequentialEvaluator
from sentence_transformers.util import cos_sim
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

from datasets import load_dataset, concatenate_datasets

In [2]:
torch.cuda.is_available()

True

In [3]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'), add_to_git_credential=True)

In [4]:
dataset=load_dataset("philschmid/finanical-rag-embedding-dataset",split="train")

README.md:   0%|          | 0.00/882 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['question', 'context'],
    num_rows: 7000
})

In [6]:
dataset["question"][10],dataset["context"][10]

('What is the H100 GPU designed to accelerate?',
 'H100 is ideal for accelerating applications such as large language models, deep recommender systems, genomics and complex digital twins.')

In [7]:
dataset=dataset.rename_column("question","anchor")
dataset=dataset.rename_column("context","positive")
dataset=dataset.add_column("id",range(len(dataset)))

In [8]:
dataset=dataset.shuffle()
dataset=dataset.train_test_split(test_size=0.1)

In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'id'],
        num_rows: 6300
    })
    test: Dataset({
        features: ['anchor', 'positive', 'id'],
        num_rows: 700
    })
})

In [10]:
dataset["train"].to_json("train_dataset.json",orient="records",lines="true")
dataset["test"].to_json("test_dataset.json",lines="true")

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

238680

Creating a base model for eval

In [29]:
model_id = "nomic-ai/modernbert-embed-base"

model = SentenceTransformer(model_id,device="cuda" if torch.cuda.is_available() else "cpu")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [30]:
test_dataset=load_dataset("json",data_files="test_dataset.json",split="train") # Define the split because else we get it as key of train,
train_dataset=load_dataset("json",data_files="train_dataset.json",split="train") #so we go a layer deeper with selecting train split


corpus_dataset=concatenate_datasets([train_dataset,test_dataset])

corpus=dict(zip(corpus_dataset["id"],corpus_dataset["positive"]))
queries=dict(zip(corpus_dataset["id"],corpus_dataset["anchor"]))


#mapping of relevant docs to their queries

relevant_docs={}

for query_id in queries:
    relevant_docs[query_id]=[query_id]

In [13]:
matryoshka_dim = [768, 512, 256, 128, 64]

matryoshka_eval=[]

for dim in matryoshka_dim:

    ir_evaluator=InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=f"dim_{dim}",
        truncate_dim=dim,
        score_functions={"cosine":cos_sim}
    )

    matryoshka_eval.append(ir_evaluator)

evaluator=SequentialEvaluator(
    evaluators=matryoshka_eval)


In [31]:
base_results=evaluator(model)

In [32]:
print(base_results)

{'dim_768_cosine_accuracy@1': 0.6398571428571429, 'dim_768_cosine_accuracy@3': 0.771, 'dim_768_cosine_accuracy@5': 0.8188571428571428, 'dim_768_cosine_accuracy@10': 0.8685714285714285, 'dim_768_cosine_precision@1': 0.6398571428571429, 'dim_768_cosine_precision@3': 0.257, 'dim_768_cosine_precision@5': 0.1637714285714286, 'dim_768_cosine_precision@10': 0.08685714285714285, 'dim_768_cosine_recall@1': 0.6398571428571429, 'dim_768_cosine_recall@3': 0.771, 'dim_768_cosine_recall@5': 0.8188571428571428, 'dim_768_cosine_recall@10': 0.8685714285714285, 'dim_768_cosine_ndcg@10': 0.7532614672892363, 'dim_768_cosine_mrr@10': 0.7164352607709747, 'dim_768_cosine_map@100': 0.720905169067066, 'dim_512_cosine_accuracy@1': 0.6364285714285715, 'dim_512_cosine_accuracy@3': 0.7682857142857142, 'dim_512_cosine_accuracy@5': 0.8147142857142857, 'dim_512_cosine_accuracy@10': 0.866, 'dim_512_cosine_precision@1': 0.6364285714285715, 'dim_512_cosine_precision@3': 0.2560952380952381, 'dim_512_cosine_precision@5': 

In [34]:
print("\nBase Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'map@100',
    'accuracy@1',
    'accuracy@3',
    'accuracy@5',
    'accuracy@10',
    'precision@1',
    'precision@3',
    'precision@5',
    'precision@10',
    'recall@1',
    'recall@3',
    'recall@5',
    'recall@10'
]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dim:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(base_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {base_results['sequential_score']:1f}")


Base Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.7533       0.7504       0.7452       0.7241       0.6777 
mrr@10                 0.7164       0.7135       0.7079       0.6857       0.6395 
map@100                0.7209       0.7181       0.7126       0.6908       0.6453 
accuracy@1             0.6399       0.6364       0.6303       0.6057       0.5599 
accuracy@3             0.7710       0.7683       0.7627       0.7443       0.6950 
accuracy@5             0.8189       0.8147       0.8084       0.7913       0.7441 
accuracy@10            0.8686       0.8660       0.8621       0.8443       0.7973 
precision@1            0.6399       0.6364       0.6303       0.6057       0.5599 
precision@3            0.2570       0.2561       0.2

In [17]:
model = SentenceTransformer(
    model_id,
    model_kwargs={"attn_implementation": "sdpa"},
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        model_name="ModernBERT Embed base Finance 10k Matryoshka",
    ),
)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

In [18]:
# model[0].auto_model.gradient_checkpointing_enable()

In [19]:
base_loss=MultipleNegativesRankingLoss(model=model)

train_loss = MatryoshkaLoss(
    model=model,
    loss=base_loss,
    matryoshka_dims=matryoshka_dim)

In [20]:
# args = SentenceTransformerTrainingArguments(
#     output_dir="modernbert-embed-finance-matryoshka", # output directory and hugging face model ID
#     num_train_epochs=4,                                        # number of epochs
#     per_device_train_batch_size=24,                            # train batch size
#     gradient_accumulation_steps=16,                            # for a global batch size of 512
#     per_device_eval_batch_size=8,                             # evaluation batch size
#     warmup_ratio=0.1,                                          # warmup ratio
#     learning_rate=2e-5,                                        # learning rate, 2e-5 is a good value
#     lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
#     optim="adamw_torch_fused",                                 # use fused adamw optimizer
#     batch_sampler=BatchSamplers.NO_DUPLICATES,                 # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
#     eval_strategy="epoch",                                     # evaluate after each epoch
#     save_strategy="epoch",                                     # save after each epoch
#     logging_steps=10,                                          # log every 10 steps
#     save_total_limit=3,                                        # save only the last 3 models
#     load_best_model_at_end=True,                               # load the best model when training ends
#     metric_for_best_model="eval_dim_128_cosine_ndcg@10",       # Optimizing for the best ndcg@10 score for the 128 dimension
#     report_to="none"                                           # Turning off training logging for now, input 'wandb' etc. if desired.
# )


In [21]:
args = SentenceTransformerTrainingArguments(
    output_dir="modernbert-embed-finance-matryoshka",
    num_train_epochs=4,
    # --- REDUCE MICRO-BATCH SIZE ---
    per_device_train_batch_size=8,
    gradient_accumulation_steps=48,
    per_device_eval_batch_size=4,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    optim="adamw_torch_fused",
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_dim_128_cosine_ndcg@10",
    report_to="none",
    fp16=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [22]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset.select_columns(["anchor","positive"]),
    loss=train_loss,
    evaluator=evaluator
)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [23]:
trainer.train()

# Save the best model based on our eval_dim_128_cosine_ndcg@10 criteria
trainer.save_model()

W0211 12:29:00.746000 1394 torch/_inductor/utils.py:1558] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Dim 768 Cosine Accuracy@1,Dim 768 Cosine Accuracy@3,Dim 768 Cosine Accuracy@5,Dim 768 Cosine Accuracy@10,Dim 768 Cosine Precision@1,Dim 768 Cosine Precision@3,Dim 768 Cosine Precision@5,Dim 768 Cosine Precision@10,Dim 768 Cosine Recall@1,Dim 768 Cosine Recall@3,Dim 768 Cosine Recall@5,Dim 768 Cosine Recall@10,Dim 768 Cosine Ndcg@10,Dim 768 Cosine Mrr@10,Dim 768 Cosine Map@100,Dim 512 Cosine Accuracy@1,Dim 512 Cosine Accuracy@3,Dim 512 Cosine Accuracy@5,Dim 512 Cosine Accuracy@10,Dim 512 Cosine Precision@1,Dim 512 Cosine Precision@3,Dim 512 Cosine Precision@5,Dim 512 Cosine Precision@10,Dim 512 Cosine Recall@1,Dim 512 Cosine Recall@3,Dim 512 Cosine Recall@5,Dim 512 Cosine Recall@10,Dim 512 Cosine Ndcg@10,Dim 512 Cosine Mrr@10,Dim 512 Cosine Map@100,Dim 256 Cosine Accuracy@1,Dim 256 Cosine Accuracy@3,Dim 256 Cosine Accuracy@5,Dim 256 Cosine Accuracy@10,Dim 256 Cosine Precision@1,Dim 256 Cosine Precision@3,Dim 256 Cosine Precision@5,Dim 256 Cosine Precision@10,Dim 256 Cosine Recall@1,Dim 256 Cosine Recall@3,Dim 256 Cosine Recall@5,Dim 256 Cosine Recall@10,Dim 256 Cosine Ndcg@10,Dim 256 Cosine Mrr@10,Dim 256 Cosine Map@100,Dim 128 Cosine Accuracy@1,Dim 128 Cosine Accuracy@3,Dim 128 Cosine Accuracy@5,Dim 128 Cosine Accuracy@10,Dim 128 Cosine Precision@1,Dim 128 Cosine Precision@3,Dim 128 Cosine Precision@5,Dim 128 Cosine Precision@10,Dim 128 Cosine Recall@1,Dim 128 Cosine Recall@3,Dim 128 Cosine Recall@5,Dim 128 Cosine Recall@10,Dim 128 Cosine Ndcg@10,Dim 128 Cosine Mrr@10,Dim 128 Cosine Map@100,Dim 64 Cosine Accuracy@1,Dim 64 Cosine Accuracy@3,Dim 64 Cosine Accuracy@5,Dim 64 Cosine Accuracy@10,Dim 64 Cosine Precision@1,Dim 64 Cosine Precision@3,Dim 64 Cosine Precision@5,Dim 64 Cosine Precision@10,Dim 64 Cosine Recall@1,Dim 64 Cosine Recall@3,Dim 64 Cosine Recall@5,Dim 64 Cosine Recall@10,Dim 64 Cosine Ndcg@10,Dim 64 Cosine Mrr@10,Dim 64 Cosine Map@100,Sequential Score
1,0.309232,No log,0.711571,0.839429,0.876857,0.915143,0.711571,0.279810,0.175371,0.091514,0.711571,0.839429,0.876857,0.915143,0.815529,0.783380,0.786705,0.710000,0.837714,0.875286,0.913286,0.710000,0.279238,0.175057,0.091329,0.710000,0.837714,0.875286,0.913286,0.813754,0.781638,0.785052,0.705000,0.834714,0.871571,0.911000,0.705000,0.278238,0.174314,0.091100,0.705000,0.834714,0.871571,0.911000,0.810381,0.777923,0.781427,0.685571,0.819571,0.859000,0.900857,0.685571,0.273190,0.171800,0.090086,0.685571,0.819571,0.859000,0.900857,0.794822,0.760681,0.764432,0.651143,0.786714,0.828857,0.878857,0.651143,0.262238,0.165771,0.087886,0.651143,0.786714,0.828857,0.878857,0.764681,0.728199,0.732388,0.764681
2,0.067463,No log,0.722000,0.849857,0.888857,0.924286,0.722000,0.283286,0.177771,0.092429,0.722000,0.849857,0.888857,0.924286,0.825734,0.793885,0.797023,0.720857,0.850714,0.887429,0.923571,0.720857,0.283571,0.177486,0.092357,0.720857,0.850714,0.887429,0.923571,0.824488,0.792497,0.795665,0.719143,0.847286,0.881571,0.920286,0.719143,0.282429,0.176314,0.092029,0.719143,0.847286,0.881571,0.920286,0.821869,0.790119,0.793494,0.695429,0.829000,0.866714,0.909714,0.695429,0.276333,0.173343,0.090971,0.695429,0.829000,0.866714,0.909714,0.804475,0.770585,0.774316,0.663143,0.799714,0.841000,0.886286,0.663143,0.266571,0.168200,0.088629,0.663143,0.799714,0.841000,0.886286,0.775675,0.740131,0.744562,0.775675
3,0.050532,No log,0.723286,0.854143,0.890143,0.926143,0.723286,0.284714,0.178029,0.092614,0.723286,0.854143,0.890143,0.926143,0.827696,0.795843,0.798986,0.722429,0.851571,0.886571,0.925286,0.722429,0.283857,0.177314,0.092529,0.722429,0.851571,0.886571,0.925286,0.825934,0.793906,0.797124,0.721714,0.849000,0.884857,0.923571,0.721714,0.283000,0.176971,0.092357,0.721714,0.849000,0.884857,0.923571,0.824281,0.792291,0.795628,0.698857,0.834143,0.871571,0.914714,0.698857,0.278048,0.174314,0.091471,0.698857,0.834143,0.871571,0.914714,0.808660,0.774530,0.778159,0.668000,0.805714,0.847286,0.894857,0.668000,0.268571,0.169457,0.089486,0.668000,0.805714,0.847286,0.8948

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
trainer.model.push_to_hub("modernbert-embed-finance-matryoshka")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pfcos1z/model.safetensors:   0%|          |  552kB /  596MB            

'https://huggingface.co/rya23/modernbert-embed-finance-matryoshka/commit/5556e437de521b876301a8b81de9b00df5ada704'

In [25]:
fine_tuned_model = SentenceTransformer(
    args.output_dir, device="cuda" if torch.cuda.is_available() else "cpu"
)

# Evaluate the model
ft_results = evaluator(fine_tuned_model)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()


In [27]:
# Print header
print("Fine Tuned Model Evaluation Results")
print("-" * 85)
print(f"{'Metric':15} {'768d':>12} {'512d':>12} {'256d':>12} {'128d':>12} {'64d':>12}")
print("-" * 85)

# List of metrics to display
metrics = [
    'ndcg@10',
    'mrr@10',
    'map@100',
    'accuracy@1',
    'accuracy@3',
    'accuracy@5',
    'accuracy@10',
    'precision@1',
    'precision@3',
    'precision@5',
    'precision@10',
    'recall@1',
    'recall@3',
    'recall@5',
    'recall@10'
]

# Print each metric
for metric in metrics:
    values = []
    for dim in matryoshka_dim:
        key = f"dim_{dim}_cosine_{metric}"
        values.append(ft_results[key])

    # Highlight NDCG@10
    metric_name = f"=={metric}==" if metric == "ndcg@10" else metric
    print(f"{metric_name:15}", end="  ")
    for val in values:
        print(f"{val:12.4f}", end=" ")
    print()

# Print sequential score
print("-" * 85)
print(f"{'seq_score:'} {ft_results['sequential_score']:1f}")

Fine Tuned Model Evaluation Results
-------------------------------------------------------------------------------------
Metric                  768d         512d         256d         128d          64d
-------------------------------------------------------------------------------------
==ndcg@10==            0.8289       0.8272       0.8255       0.8110       0.7831 
mrr@10                 0.7970       0.7951       0.7937       0.7770       0.7469 
map@100                0.8001       0.7983       0.7970       0.7806       0.7511 
accuracy@1             0.7247       0.7239       0.7230       0.7024       0.6694 
accuracy@3             0.8551       0.8536       0.8510       0.8360       0.8053 
accuracy@5             0.8901       0.8876       0.8864       0.8730       0.8487 
accuracy@10            0.9276       0.9269       0.9241       0.9164       0.8960 
precision@1            0.7247       0.7239       0.7230       0.7024       0.6694 
precision@3            0.2850       0.2845     

In [36]:
print("Base vs Fine-Tuned Comparison")
print("-" * 130)
print(f"{'Metric':15} {'Dim':>6} {'Base':>12} {'FT':>12} {'Δ Abs':>12} {'Δ %':>12}")
print("-" * 130)

for metric in metrics:
    for dim in matryoshka_dim:
        key = f"dim_{dim}_cosine_{metric}"

        base_val = base_results[key]
        ft_val = ft_results[key]

        abs_gain = ft_val - base_val
        pct_gain = (abs_gain / base_val * 100) if base_val != 0 else 0.0

        metric_name = f"=={metric}==" if metric == "ndcg@10" else metric

        print(
            f"{metric_name:15} "
            f"{dim:6} "
            f"{base_val:12.4f} "
            f"{ft_val:12.4f} "
            f"{abs_gain:12.4f} "
            f"{pct_gain:11.2f}%"
        )

print("-" * 130)

# sequential score comparison
seq_base = base_results["sequential_score"]
seq_ft = ft_results["sequential_score"]
seq_abs = seq_ft - seq_base
seq_pct = (seq_abs / seq_base * 100) if seq_base != 0 else 0.0

print(f"{'seq_score':15} {'ALL':6} {seq_base:12.4f} {seq_ft:12.4f} {seq_abs:12.4f} {seq_pct:11.2f}%")

Base vs Fine-Tuned Comparison
----------------------------------------------------------------------------------------------------------------------------------
Metric             Dim         Base           FT        Δ Abs          Δ %
----------------------------------------------------------------------------------------------------------------------------------
==ndcg@10==        768       0.7533       0.8289       0.0756       10.04%
==ndcg@10==        512       0.7504       0.8272       0.0769       10.24%
==ndcg@10==        256       0.7452       0.8255       0.0803       10.78%
==ndcg@10==        128       0.7241       0.8110       0.0869       12.00%
==ndcg@10==         64       0.6777       0.7831       0.1054       15.56%
mrr@10             768       0.7164       0.7970       0.0805       11.24%
mrr@10             512       0.7135       0.7951       0.0817       11.45%
mrr@10             256       0.7079       0.7937       0.0858       12.12%
mrr@10             128       0.68